# 01_text_preprocessing: Cleaning, Normalization, Stemming, and Subword Simulation

This notebook implements classical text preprocessing steps (Porter stemming and WordNet lemmatization) using NLTK, and simulates a basic Byte-Pair Encoding (BPE) subword merge loop.


In [1]:
import nltk
import re
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# 1. Download required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

raw_text = "The cats were studying studying studies in Seattle's libraries! https://example.com"

# 2. Basic regex cleaning
cleaned_text = re.sub(r"https?://\S+", "", raw_text)
cleaned_text = re.sub(r"[^\w\s]", "", cleaned_text).lower()
print("Cleaned Text:", cleaned_text)

# 3. Tokenize
tokens = word_tokenize(cleaned_text)
print("Tokens:", tokens)

# 4. Stemming vs Lemmatization
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stemmed = [stemmer.stem(t) for t in tokens]
lemmatized = [lemmatizer.lemmatize(t, pos='v') for t in tokens]

print("\nLexical Reduction Comparison:")
print(f"{'Original':<12} | {'Stemmed':<12} | {'Lemmatized':<12}")
print("-" * 42)
for o, s, l in zip(tokens, stemmed, lemmatized):
    print(f"{o:<12} | {s:<12} | {l:<12}")


Cleaned Text: the cats were studying studying studies in seattles libraries 
Tokens: ['the', 'cats', 'were', 'studying', 'studying', 'studies', 'in', 'seattles', 'libraries']



Lexical Reduction Comparison:
Original     | Stemmed      | Lemmatized  
------------------------------------------
the          | the          | the         
cats         | cat          | cat         
were         | were         | be          
studying     | studi        | study       
studying     | studi        | study       
studies      | studi        | study       
in           | in           | in          
seattles     | seattl       | seattles    
libraries    | librari      | libraries   


## Byte-Pair Encoding (BPE) Simulation
Let's simulate a basic bottom-up BPE tokenizer training merge loop on a tiny vocabulary.


In [2]:
from collections import Counter, defaultdict

# Tiny BPE training corpus
corpus = {
    "l o w _": 5,
    "l o w e r _": 2,
    "n e w e s t _": 6
}

def get_stats(corpus):
    pairs = defaultdict(int)
    for word, freq in corpus.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, corpus):
    new_corpus = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in corpus:
        w_new = p.sub(''.join(pair), word)
        new_corpus[w_new] = corpus[word]
    return new_corpus

# Run 5 BPE merge iterations
vocab = set("l o w e r n s t _".split())
print("Initial Vocab:", sorted(vocab))

for i in range(5):
    pairs = get_stats(corpus)
    if not pairs:
        break
    best_pair = max(pairs, key=pairs.get)
    corpus = merge_vocab(best_pair, corpus)
    merged_token = ''.join(best_pair)
    vocab.add(merged_token)
    print(f"\nIteration {i+1}: Merging {best_pair} (frequency={pairs[best_pair]})")
    print("Updated Corpus State:", corpus)

print("\nFinal BPE Vocab:", sorted(vocab))


Initial Vocab: ['_', 'e', 'l', 'n', 'o', 'r', 's', 't', 'w']

Iteration 1: Merging ('w', 'e') (frequency=8)
Updated Corpus State: {'l o w _': 5, 'l o we r _': 2, 'n e we s t _': 6}

Iteration 2: Merging ('l', 'o') (frequency=7)
Updated Corpus State: {'lo w _': 5, 'lo we r _': 2, 'n e we s t _': 6}

Iteration 3: Merging ('n', 'e') (frequency=6)
Updated Corpus State: {'lo w _': 5, 'lo we r _': 2, 'ne we s t _': 6}

Iteration 4: Merging ('ne', 'we') (frequency=6)
Updated Corpus State: {'lo w _': 5, 'lo we r _': 2, 'newe s t _': 6}

Iteration 5: Merging ('newe', 's') (frequency=6)
Updated Corpus State: {'lo w _': 5, 'lo we r _': 2, 'newes t _': 6}

Final BPE Vocab: ['_', 'e', 'l', 'lo', 'n', 'ne', 'newe', 'newes', 'o', 'r', 's', 't', 'w', 'we']


### Output Explanation
- **Lexical Reduction**: Stemming cuts off suffixes heuristically (e.g. `"studying"` $ightarrow$ `"studi"`), whereas lemmatization resolves tokens to morphological base forms using grammatical tagging dictionary lookups (e.g. `"studies"` $ightarrow$ `"study"`).
- **BPE merges**: The simulator finds adjacent character pairs (e.g., `(e, s)` then `(es, t)`) and groups them into single, multi-character subwords.
